# Calculation of skills RCA with ESCO data
Felix Zaussinger | 09.12.2022

**Core Analysis Goal(s)**
1. Create function for estimating RCA based on bipartite network.
2. Apply to ESCO skills data.
3. Compare to output from COOC approach.

**Key Insight(s)**
1.
2.

**Sources**
1. [https://notes.quantecon.org/submission/5b32e9b0b9eab00015b89f7d](https://notes.quantecon.org/submission/5b32e9b0b9eab00015b89f7d)

In [3]:
import os
import sys
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from src import utils, plotting_utils, stats_utils
import mapping_career_causeways

# OPTIONAL: Load the "autoreload" extension so that code can change
%load_ext autoreload

# OPTIONAL: always reload modules so that as you change code in src, it gets loaded
%autoreload 2

# Settings
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

sns.set_context("notebook")
# sns.set(rc={'figure.figsize': (16, 9.)})
sns.set_style("ticks")

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 120)
pd.options.display.float_format = "{:,.2f}".format

logging.basicConfig(level=logging.INFO, stream=sys.stdout)

# load paths
useful_paths = utils.UsefulPaths()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Read data

In [2]:
from src.data.framework import Esco
from src.modelling import occupation_distance as occdist

# esco data
esco = Esco()
occdist.occ_sim_matrix_by_levels()

,,,concept_uri,http://data.europa.eu/esco/occupation/00030d09-2b3a-4efd-87cc-c4ea39d27c34,http://data.europa.eu/esco/occupation/000e93a3-d956-4e45-aacb-f12c83fedf84,http://data.europa.eu/esco/occupation/0019b951-c699-4191-8208-9822882d150c,http://data.europa.eu/esco/occupation/0022f466-426c-41a4-ac96-a235c945cf97,http://data.europa.eu/esco/occupation/002da35b-7808-43f3-83bf-63596b8b351f,http://data.europa.eu/esco/occupation/0044c991-c26f-4261-a213-4bd1c0564a4c,http://data.europa.eu/esco/occupation/00634fc4-802a-461b-8af0-499273756f99,http://data.europa.eu/esco/occupation/00674f21-2f8f-4a41-9896-133f7cbe2a6e,http://data.europa.eu/esco/occupation/006cc1f9-2841-41c3-991a-dc3f2f3bd533,http://data.europa.eu/esco/occupation/00747307-6341-4266-a061-25416f7c6a96,http://data.europa.eu/esco/occupation/0085f271-5999-4d42-b879-75c3b4269883,http://data.europa.eu/esco/occupation/0090dd54-057a-4f65-bbe8-f74c4999e8e8,http://data.europa.eu/esco/occupation/00913533-5237-4870-9003-d1cb806603c9,http://data.europa.eu/esco/occupation/009d29de-5872-43be-8d9b-abd27f8c99f1,http://data.europa.eu/esco/occupation/00ab29a3-6d4a-4df9-b46d-de31069e36e8,http://data.europa.eu/esco/occupation/00ab5610-e715-428f-99f6-b1e5e469dbcd,http://data.europa.eu/esco/occupation/00c8d1cf-24f2-47d1-9f10-765e1c102811,http://data.europa.eu/esco/occupation/00cee175-1376-43fb-9f02-ba3d7a910a58,http://data.europa.eu/esco/occupation/00fdad73-4551-4e43-b256-1d4a3759db1e,http://data.europa.eu/esco/occupation/01205776-db8d-4588-a6a6-56d0b9c660a9,http://data.europa.eu/esco/occupation/01484951-15e6-4b88-a20f-1201868d36a0,http://data.europa.eu/esco/occupation/0152b59d-5005-470c-9273-ef321477d8dc,http://data.europa.eu/esco/occupation/01649ae9-22ee-4dba-83a8-3a2c51fbbe95,http://data.europa.eu/esco/occupation/0181835d-dd00-49e2-9fc4-e1382a65957f,http://data.europa.eu/esco/occupation/01989dd8-36c0-43cc-a82d-2bfc409de9c4,http://data.europa.eu/esco/occupation/01a83071-6533-4675-b2da-f0c6d02669f5,http://data.europa.eu/esco/occupation/01bcac9f-7881-4664-bd63-dea700e3b117,http://data.europa.eu/esco/occupation/01da600c-ddb3-4a37-95e9-5301b459ae90,http://data.europa.eu/esco/occupation/01ffb917-98dc-48c1-91ad-93c4104e791d,http://data.europa.eu/esco/occupation/021663ca-a367-472e-a1f0-6d8ab5b2d860,http://data.europa.eu/esco/occupation/02375132-3290-4221-bf55-d93ec61a3bf3,http://data.europa.eu/esco/occupation/02447817-ea01-4d8b-b09c-8bc128e447e6,http://data.europa.eu/esco/occupation/0249b723-1d20-4ff9-8623-d20b98d0b4a4,http://data.europa.eu/esco/occupation/02609ece-cea3-43fa-bf38-b6fb821f28b2,http://data.europa.eu/esco/occupation/026401a9-7d56-4d7b-8052-38985ddbeafa,http://data.europa.eu/esco/occupation/0264b794-3771-47aa-a675-78bf13b62ee7,http://data.europa.eu/esco/occupation/026a5233-cad9-4b1d-8852-22c2685ab34e,http://data.europa.eu/esco/occupation/029a5b78-4d1c-4cf9-a2d2-bb8d8729b8a3,http://data.europa.eu/esco/occupation/02b5b7b0-b856-4ac7-924c-8c35c7880b2a,http://data.europa.eu/esco/occupation/02bd7cac-8263-4425-b03d-19001ffe8787,http://data.europa.eu/esco/occupation/02d4f153-8e43-444d-8bd4-8171d49eab12,http://data.europa.eu/esco/occupation/02e19186-458f-40eb-9e78-31c656d12ac7,http://data.europa.eu/esco/occupation/02e94cd6-8e6d-4e9b-b695-edb460c70528,http://data.europa.eu/esco/occupation/02eb0ae6-ecdd-4602-9c8e-60ffe6dbe1e2,http://data.europa.eu/esco/occupation/02f1ae71-f89c-47b7-af2c-50d7ab1d9b8a,http://data.europa.eu/esco/occupation/02f26b2f-879c-4811-8381-65e8dd3f4a16,http://data.europa.eu/esco/occupation/02ff199e-38b3-45fd-b659-68846cfeeb92,http://data.europa.eu/esco/occupation/032d4115-03ec-49c9-853d-678f8a41969b,http://data.europa.eu/esco/occupation/034cad59-e666-4770-a7ef-a337257f8072,http://data.europa.eu/esco/occupation/03632d98-0ae3-4dd2-941c-3b48de9a0219,http://data.europa.eu/esco/occupation/0368c4d4-94cf-4168-a84d-283a25880e0d,http://data.europa.eu/esco/occupation/036dc42f-bd4d-4533-a3cc-fcd313227ce3,http://data.europa.eu/esco/occupation/03b2f112-a891-461d-a8d4-effb9dcf2df6,http://dat

## Preprocessing

Let's focus on an application of network analysis that is applied to international trade data to replicate some of the results contained in the Hidalgo (2007) paper and later in the The Atlas of Complexity and The Observatory of Economic Complexity.

The Hidalgo (2007) paper is used as a motivating example to demonstrate various tools that are available in the Python ecosystem.

In this setting we want to looking at a characterisation of International Trade data by considering:

    Nodes: Products
    Edges: the likelihood of two products being co-exported

Assumption: If products are highly co-exported across countries, then the products are revealed to be more likely to share similar factors of production (or capabilities) required to produce them. For example, Shirts and Pants require a set of similar skills that lend themselves to be co-exported, while shirts and cars are much more dissimilar.

This relational information between products can be represented by a edge weights.

A high value means they have a high likelihood of being co-exported :

Coexport Probability: Min{P(P1 | P2), P(P2 | P1)}

##### Revealed Comparative Advantage and Mcp matrices

The literature uses the standard Balassa definition for Revealed Comparative Advantage

RCAcpt = Ecpt / Ect / Ept/ Et

where,

    Ecpt

are exports from country c in product p at time t
Ect
are total country c exports at time t
Ept
are total product p exports at time t
Et
are total world exports at time t

Reference: Balassa, B. (1965), Trade Liberalisation and Revealed Comparative Advantage, The Manchester School, 33, 99-123.

To compute RCA we need to aggregate data at difference levels to obtain each component of the fraction defined above.

Let's break the equation down to figure out what needs to be computed:
Ect=∑pEcpt

##### Compute Proximity Matrices (ϕpp′)

Proximity: A high proximity value suggests any two products are exported by a similar set of countries.

ϕij=min{P(RCAi>=1|RCAj>=1),P(RCAj>=1|RCAi>=1)}

## Modelling

## Visualisation